ML Modelling
1. we have covered whats happening currently with the data
2. feature engineering and ML modelling helps cover understand what can happen in the future

In [0]:
%sql
CREATE OR REPLACE TABLE workforce.gold.ml_employee_features AS

SELECT

    employee_id,

    age,

    distance_from_home,

    education,

    environment_satisfaction,

    job_involvement,

    job_level,

    job_satisfaction,

    monthly_income,

    num_companies_worked,

    percent_salary_hike,

    relationship_satisfaction,

    stock_option_level,

    total_working_years,

    training_times_last_year,

    work_life_balance,

    years_at_company,

    years_in_current_role,

    years_since_last_promotion,

    years_with_curr_manager,

    over_time,

    attrition

FROM workforce.silver.employee

In [0]:
feature_df = spark.table(
    "workforce.gold.ml_employee_features"
)

display(feature_df)

In [0]:
display(
    feature_df.groupBy(
        "attrition"
    ).count()
)

Convert to Pandas

In [0]:
pdf=feature_df.toPandas()

Split Features & Target

In [0]:
X = pdf.drop(
    columns=[
        "employee_id",
        "attrition"
    ]
)

y = pdf["attrition"]

Train Test Split

In [0]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(

    X,
    y,

    test_size=0.2,

    random_state=42,

    stratify=y

)

In [0]:
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier(

    n_estimators=200,

    random_state=42

)

model.fit(
    X_train,
    y_train
)

In [0]:
predictions = model.predict(
    X_test
)

Evaluation

In [0]:
from sklearn.metrics import *

print(

    classification_report(
        y_test,
        predictions
    )

)

In [0]:
print(

    roc_auc_score(
        y_test,
        model.predict_proba(X_test)[:,1]
    )

)

Feature Importance

In [0]:
import pandas as pd

importance_df = pd.DataFrame({

    "feature": X.columns,

    "importance": model.feature_importances_

})

importance_df = (

    importance_df

    .sort_values(
        "importance",
        ascending=False
    )

)

display(
    importance_df
)

Predictions Table

In [0]:
pdf["attrition_probability"] = (

    model.predict_proba(X)[:,1]

)

In [0]:
prediction_df = spark.createDataFrame(pdf)

In [0]:
(
    prediction_df

    .write

    .format("delta")

    .mode("overwrite")

    .saveAsTable(
        "workforce.gold.fct_attrition_predictions"
    )
)